# מבוא ללמידה עמוקה – מטלת בית 3  
## Afeka College of Engineering  
### Semester B, 2026

**סטודנטים:**  
- אביעד צפרי – 322281452  
- דוד אזימוב – 205697592  

---

## Question 1

Consider an LSTM network processing a sequence of length $T = 5$:

**a)** Write the complete mathematical formulation for the forget gate $(f_t)$, input gate $(i_t)$, cell state $(C_t)$, and output gate $(o_t)$ at any time step $t$.

**b)** Explain mathematically how LSTM addresses the vanishing gradient problem that occurs in simple RNNs. Support your explanation with derivatives.

**c)** If we have a batch size of 32, input dimension of 100, and hidden state dimension of 64, calculate:

* The dimension of each weight matrix
* The total number of parameters
* The memory requirements during training

---

### Answer 1

An LSTM receives an input sequence of length $T = 5$.  
At each time step $t$, the network receives the current input vector $x_t$, the previous hidden state $h_{t-1}$, and the previous cell state $C_{t-1}$.

The purpose of the LSTM gates is to control what information should be forgotten, what new information should be stored, and what information should be passed forward as the hidden state.

#### a)

For a sequence of length $T = 5$, the LSTM performs the following computations for each time step:

$$
t = 1,2,3,4,5
$$

Let the input vector at time step $t$ be:

$$
x_t \in \mathbb{R}^{d}
$$

and let the hidden state and cell state be:

$$
h_t \in \mathbb{R}^{h}, \qquad C_t \in \mathbb{R}^{h}
$$

where $d$ is the input dimension and $h$ is the hidden dimension. The initial states $h_0$ and $C_0$ are either learned, given, or commonly initialized to zero.

The forget gate is:

$$
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
$$

where $f_t \in \mathbb{R}^{h}$. This gate controls how much information from the previous cell state $C_{t-1}$ is preserved.

The input gate is:

$$
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
$$

where $i_t \in \mathbb{R}^{h}$. This gate controls how much new information is written into the cell state.

The candidate cell state is:

$$
\tilde{C}_t = \tanh(W_C x_t + U_C h_{t-1} + b_C)
$$

where $\tilde{C}_t \in \mathbb{R}^{h}$. This vector contains the new candidate information that may be added to the memory.

The updated cell state is:

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

The first term, $f_t \odot C_{t-1}$, keeps part of the previous memory.  
The second term, $i_t \odot \tilde{C}_t$, adds new information to the memory.

The output gate is:

$$
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
$$

where $o_t \in \mathbb{R}^{h}$. This gate controls how much of the cell state is exposed as the hidden state.

Finally, the hidden state is computed as:

$$
h_t = o_t \odot \tanh(C_t)
$$

For each gate $g \in \{f,i,C,o\}$, the parameters have the general dimensions:

$$
W_g \in \mathbb{R}^{h \times d}, \qquad
U_g \in \mathbb{R}^{h \times h}, \qquad
b_g \in \mathbb{R}^{h}
$$

where $W_g$ multiplies the current input $x_t$, $U_g$ multiplies the previous hidden state $h_{t-1}$, and $b_g$ is the bias vector.

#### b)

In a simple RNN, the hidden state is usually written as:

$$
h_t = \phi(W_{hh}h_{t-1} + W_{xh}x_t + b_h)
$$

where $\phi$ is usually a nonlinear activation function such as $\tanh$.

When backpropagation through time is applied, the gradient of the loss with respect to an earlier hidden state $h_k$ depends on a product of many Jacobian matrices:

$$
\frac{\partial L}{\partial h_k}
=
\frac{\partial L}{\partial h_T}
\prod_{t=k+1}^{T}
\frac{\partial h_t}{\partial h_{t-1}}
$$

For a simple RNN:

$$
\frac{\partial h_t}{\partial h_{t-1}}
=
\text{diag}(\phi'(a_t))W_{hh}
$$

where:

$$
a_t = W_{hh}h_{t-1} + W_{xh}x_t + b_h
$$

Therefore:

$$
\frac{\partial L}{\partial h_k}
=
\frac{\partial L}{\partial h_T}
\prod_{t=k+1}^{T}
\text{diag}(\phi'(a_t))W_{hh}
$$

This repeated multiplication is the source of the vanishing gradient problem. If the spectral norm of the recurrent Jacobian is smaller than $1$, then the product becomes smaller and smaller as the distance between $k$ and $T$ increases:

$$
\left|
\frac{\partial L}{\partial h_k}
\right|
\leq
\left|
\frac{\partial L}{\partial h_T}
\right|
\prod_{t=k+1}^{T}
\left|
\text{diag}(\phi'(a_t))W_{hh}
\right|
$$

If:

$$
\left|
\text{diag}(\phi'(a_t))W_{hh}
\right| < 1
$$

then:

$$
\prod_{t=k+1}^{T}
\left|
\text{diag}(\phi'(a_t))W_{hh}
\right|
\rightarrow 0
$$

as the sequence length increases. This means that early time steps receive almost no useful gradient signal.

LSTM addresses this problem by introducing the cell state update:

$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t
$$

The key point is that the cell state has a direct additive path from $C_{t-1}$ to $C_t$. The derivative of the current cell state with respect to the previous cell state along this direct path is:

$$
\frac{\partial C_t}{\partial C_{t-1}} = f_t
$$

or, in vector form:

$$
\frac{\partial C_t}{\partial C_{t-1}} = \text{diag}(f_t)
$$

Therefore, across multiple time steps:

$$
\frac{\partial C_T}{\partial C_k}
=
\prod_{t=k+1}^{T}
\text{diag}(f_t)
$$

For each cell component $j$:

$$
\frac{\partial C_{T,j}}{\partial C_{k,j}}
=
\prod_{t=k+1}^{T}
f_{t,j}
$$

Since the forget gate values are produced by a sigmoid function,

$$
0 < f_{t,j} < 1
$$

the LSTM can learn to set $f_{t,j}$ close to $1$ when information should be preserved. In that case:

$$
\prod_{t=k+1}^{T}
f_{t,j}
\approx 1
$$

and the gradient can flow backward through many time steps without vanishing quickly.

Therefore, unlike a simple RNN, where gradients repeatedly pass through nonlinear recurrent transformations, the LSTM provides a nearly linear memory path through the cell state. The forget gate controls this path, allowing the model to preserve long-term dependencies when needed.


#### c)

We are given:

$$
T = 5, \qquad \text{batch size} = 32, \qquad d = 100, \qquad h = 64
$$

where $d$ is the input dimension and $h$ is the hidden dimension.

* **Weight matrix dimensions**

For each LSTM component $g \in \{f,i,C,o\}$, the parameters are:

$$
W_g \in \mathbb{R}^{h \times d}
$$

$$
U_g \in \mathbb{R}^{h \times h}
$$

$$
b_g \in \mathbb{R}^{h}
$$

Substituting $d = 100$ and $h = 64$:

$$
\boxed{
W_g \in \mathbb{R}^{64 \times 100}, \qquad
U_g \in \mathbb{R}^{64 \times 64}, \qquad
b_g \in \mathbb{R}^{64}
}
$$

* **Number of parameters**

Therefore, for each gate:

$$
\#W_g = 64 \cdot 100 = 6400
$$

$$
\#U_g = 64 \cdot 64 = 4096
$$

$$
\#b_g = 64
$$

So the number of parameters per gate is:

$$
6400 + 4096 + 64 = 10560
$$

Since the LSTM has four components: forget gate, input gate, candidate cell state, and output gate, the total number of trainable parameters is:

$$
4 \cdot 10560 = 42240
$$

Therefore:

$$
\boxed{42240}
$$

trainable parameters are required.

* **Memory requirement**

Assuming 32-bit floating point numbers (Floats), each parameter requires:

$$
4 \text{ [bytes]}
$$

Therefore, parameter memory is:

$$
42240 \cdot 4 = 168960 \text{ [bytes]}
$$

$$
\approx 165 \text{ [KB]}
$$

During training, gradients for the parameters must also be stored, so parameter and gradient memory together is:

$$
2 \cdot 168960 = 337920 \text{ [bytes]}
$$

$$
\approx 330 \text{ [KB]}
$$

Now we estimate the activation memory stored for backpropagation through time.

The input sequence contains:

$$
32 \cdot 5 \cdot 100 = 16000 \text{ floats}
$$

The hidden states contain:

$$
32 \cdot 5 \cdot 64 = 10240 \text{ floats}
$$

The cell states contain:

$$
32 \cdot 5 \cdot 64 = 10240 \text{ floats}
$$

The four gate-related activations contain:

$$
4 \cdot 32 \cdot 5 \cdot 64 = 40960 \text{ floats}
$$

Therefore, the total number of stored activation values is approximately:

$$
16000 + 10240 + 10240 + 40960 = 77440 \text{ floats}
$$

Using 32-bit floating point numbers, the activation memory is:

$$
77440 \cdot 4 = 309760 \text{ [bytes]}
$$

$$
\approx 302.5 \text{ [KB]}
$$

Therefore, a basic estimate of the memory required during training is:

$$
\text{parameters} + \text{gradients} + \text{activations}
$$

$$
= 168960 + 168960 + 309760
$$

$$
= 647680 \text{ [bytes]}
$$

$$
\approx 632.5 \text{ [KB]}
$$

$$
\boxed{ \approx 0.62 \text{ [MB]} }
$$

This is a simplified training-memory estimate. In practice, the actual memory may be larger because deep learning frameworks can store additional intermediate tensors, optimizer states, and implementation-specific buffers.

---

### Question 2

For a CNN with the following architecture:

- Input: $32 \times 32 \times 3$
- Conv1: 16 filters of size $5 \times 5$, stride 1
- MaxPool1: $2 \times 2$, stride 2
- Conv2: 32 filters of size $3 \times 3$, stride 1
- MaxPool2: $2 \times 2$, stride 2
- Fully connected: 128 neurons
- Output: 10 classes

a) Calculate the output dimensions after each layer.

b) Derive the total number of trainable parameters.

c) If we use batch normalization after Conv1, write the mathematical equations for:

- Forward pass normalization
- Parameter update during backpropagation

---

### Answer 2

Since no padding is specified in the question, we assume **valid convolution**, meaning: $P = 0$

For convolution and pooling layers, the spatial output size is calculated by:

$$
\text{Output size} =
\left\lfloor
\frac{N + 2P - F}{S}
\right\rfloor + 1
$$

where:

$$
N = \text{input size}
$$

$$
F = \text{filter/kernel size}
$$

$$
S = \text{stride}
$$

$$
P = \text{padding}
$$

#### a) Output dimentions after each layer

* **Input layer**

The input image has dimensions:

$$
32 \times 32 \times 3
$$

So the input has height $32$, width $32$, and $3$ channels (RGB).

* **Conv1**

Conv1 has $16$ filters of size $5 \times 5$, stride $1$, and no padding.

The spatial output size is:

$$
\frac{32 - 5}{1} + 1 = 28
$$

Therefore, the output after Conv1 is:

$$
28 \times 28 \times 16
$$

The depth is $16$ because Conv1 has $16$ filters.

* **MaxPool1**

MaxPool1 uses a $2 \times 2$ pooling window with stride $2$.

The spatial output size is:

$$
\frac{28 - 2}{2} + 1 = 14
$$

Therefore, the output after MaxPool1 is:

$$
14 \times 14 \times 16
$$

The depth remains $16$ because pooling changes only the spatial dimensions, not the number of channels.

* **Conv2**

Conv2 has $32$ filters of size $3 \times 3$, stride $1$, and no padding.

The spatial output size is:

$$
\frac{14 - 3}{1} + 1 = 12
$$

Therefore, the output after Conv2 is:

$$
12 \times 12 \times 32
$$

The depth is $32$ because Conv2 has $32$ filters.

* **MaxPool2**

MaxPool2 uses a $2 \times 2$ pooling window with stride $2$.

The spatial output size is:

$$
\frac{12 - 2}{2} + 1 = 6
$$

Therefore, the output after MaxPool2 is:

$$
6 \times 6 \times 32
$$

* **Flatten layer**

Before entering the fully connected layer, the feature map is flattened:

$$
6 \cdot 6 \cdot 32 = 1152
$$

Therefore, the flattened vector has dimension:

$$
1152
$$

* **Fully connected layer**

The fully connected layer has $128$ neurons, so its output dimension is:

$$
128
$$

* **Output layer**

The output layer has $10$ classes, so its output dimension is:

$$
10
$$

Therefore, the layer dimensions are:

$$
32 \times 32 \times 3
\rightarrow
28 \times 28 \times 16
\rightarrow
14 \times 14 \times 16
\rightarrow
12 \times 12 \times 32
\rightarrow
6 \times 6 \times 32
\rightarrow
1152
\rightarrow
128
\rightarrow
10
$$

#### b) Number of trainable params

* **Conv1 parameters**

Conv1 has $16$ filters, each of size $5 \times 5$.  
Since the input has $3$ channels, each filter has size:

$$
5 \times 5 \times 3
$$

Therefore, the number of weights in one Conv1 filter is:

$$
5 \cdot 5 \cdot 3 = 75
$$

Each filter also has one bias term, so the number of parameters per filter is:

$$
75 + 1 = 76
$$

Since Conv1 has $16$ filters, the total number of trainable parameters in Conv1 is:

$$
16 \cdot 76 = 1216
$$

* **MaxPool1 parameters**

MaxPool1 is a pooling layer. It does not contain weights or biases.

Therefore:

$$
\text{MaxPool1 parameters} = 0
$$

* **Conv2 parameters**

Conv2 has $32$ filters, each of size $3 \times 3$.  
The input to Conv2 is the output of MaxPool1, whose depth is $16$.

Therefore, each Conv2 filter has size:

$$
3 \times 3 \times 16
$$

The number of weights in one Conv2 filter is:

$$
3 \cdot 3 \cdot 16 = 144
$$

Each filter also has one bias term, so the number of parameters per filter is:

$$
144 + 1 = 145
$$

Since Conv2 has $32$ filters, the total number of trainable parameters in Conv2 is:

$$
32 \cdot 145 = 4640
$$

* **MaxPool2 parameters**

MaxPool2 is also a pooling layer. It does not contain weights or biases.

Therefore:

$$
\text{MaxPool2 parameters} = 0
$$

* **Fully connected layer parameters**

After MaxPool2, the output dimension is:

$$
6 \times 6 \times 32
$$

Before entering the fully connected layer, this tensor is flattened:

$$
6 \cdot 6 \cdot 32 = 1152
$$

The fully connected layer has $128$ neurons. Therefore, the number of weights is:

$$
1152 \cdot 128 = 147456
$$

Each neuron has one bias term, so there are:

$$
128
$$

bias parameters.

Therefore, the total number of trainable parameters in the fully connected layer is:

$$
147456 + 128 = 147584
$$

* **Output layer parameters**

The fully connected layer outputs $128$ values, and the output layer has $10$ classes.

Therefore, the number of weights is:

$$
128 \cdot 10 = 1280
$$

Each output neuron has one bias term, so there are:

$$
10
$$

bias parameters.

Therefore, the total number of trainable parameters in the output layer is:

$$
1280 + 10 = 1290
$$

* **Total number of trainable parameters**

The total number of trainable parameters in our CNN is:

$$
1216 + 0 + 4640 + 0 + 147584 + 1290
$$

$$
= \boxed{154730}
$$

trainable parameters.

#### c)

Batch normalization is applied after Conv1.

From part (a), the output of Conv1 is:

$$
28 \times 28 \times 16
$$

With batch size $32$, the Conv1 activation tensor has shape:

$$
32 \times 28 \times 28 \times 16
$$

For convolutional layers, batch normalization is usually applied independently for each output channel.  
Therefore, for each channel $k$, where:

$$
k = 1,2,\dots,16
$$

the mean and variance are computed over all examples in the mini-batch and over all spatial positions.

The number of values used for each channel is:

$$
m = 32 \cdot 28 \cdot 28 = 25088
$$

Let $z_{n,i,j,k}$ denote the Conv1 activation of example $n$, spatial position $(i,j)$, and channel $k$.

* **Forward pass normalization**

First, compute the mini-batch mean for channel $k$:

$$
\mu_k =
\frac{1}{m}
\sum_{n=1}^{32}
\sum_{i=1}^{28}
\sum_{j=1}^{28}
z_{n,i,j,k}
$$

Then compute the mini-batch variance for channel $k$:

$$
\sigma_k^2 =
\frac{1}{m}
\sum_{n=1}^{32}
\sum_{i=1}^{28}
\sum_{j=1}^{28}
\left(z_{n,i,j,k} - \mu_k\right)^2
$$

Normalize the activation:

$$
\hat{z}_{n,i,j,k}
=
\frac{z_{n,i,j,k} - \mu_k}
{\sqrt{\sigma_k^2 + \epsilon}}
$$

where $\epsilon$ is a small constant used for numerical stability.

Then apply the learnable scale and shift parameters:

$$
y_{n,i,j,k}
=
\gamma_k \hat{z}_{n,i,j,k} + \beta_k
$$

where $\gamma_k$ and $\beta_k$ are trainable parameters for channel $k$.

Since Conv1 has $16$ output channels, batch normalization introduces

$16$ scale parameters and $16$ shift parameters.

Therefore, batch normalization after Conv1 adds $16 + 16 = 32$ trainable parameters.

If batch normalization is included after Conv1, the total number of trainable parameters becomes:

$$
154730 + 32 = 154762
$$

* **Parameter update during backpropagation**

Let:

$$
\delta_{n,i,j,k}
=
\frac{\partial L}{\partial y_{n,i,j,k}}
$$

be the gradient arriving from the next layer.

The gradient with respect to the shift parameter is:

$$
\frac{\partial L}{\partial \beta_k}
=
\sum_{n=1}^{32}
\sum_{i=1}^{28}
\sum_{j=1}^{28}
\delta_{n,i,j,k}
$$

The gradient with respect to the scale parameter is:

$$
\frac{\partial L}{\partial \gamma_k}
=
\sum_{n=1}^{32}
\sum_{i=1}^{28}
\sum_{j=1}^{28}
\delta_{n,i,j,k}
\hat{z}_{n,i,j,k}
$$

Using gradient descent with learning rate $\eta$, the batch-normalization parameters are updated as:

$$
\gamma_k
\leftarrow
\gamma_k
-
\eta
\frac{\partial L}{\partial \gamma_k}
$$

$$
\beta_k
\leftarrow
\beta_k
-
\eta
\frac{\partial L}{\partial \beta_k}
$$

for each channel: $k = 1,2,\dots,16$

To complete the backpropagation through batch normalization, the gradient must also be propagated to the Conv1 activations $z_{n,i,j,k}$.

For each channel $k$, the compact batch-normalization backward formula is:

$$
\frac{\partial L}{\partial z_{n,i,j,k}}
=
\frac{1}{m}
\frac{\gamma_k}{\sqrt{\sigma_k^2 + \epsilon}}
\left[
m\delta_{n,i,j,k}
-
\sum_{n,i,j}\delta_{n,i,j,k}
-
\hat{z}_{n,i,j,k}
\sum_{n,i,j}
\delta_{n,i,j,k}\hat{z}_{n,i,j,k}
\right]
$$

where all summations are over the mini-batch and spatial positions for the same channel $k$.

This gradient is then passed backward into Conv1, where it is used to update the Conv1 filter weights and biases.

The running mean and running variance used during inference are not trainable parameters. They are updated during training using moving averages, not by gradient descent.

---

### Question 3

Given a training process with the following metrics:

- Training accuracy: $98\%$
- Validation accuracy: $85\%$
- Training loss: $0.02$
- Validation loss: $0.4$

a) For each of these scenarios, explain mathematically what's happening and propose solutions:

- Learning rate is too high
- Learning rate is too small
- Batch size is too high
- Batch size is too small

b) Derive the relationship between:

- Precision and Recall
- F1 score and Accuracy

---

### Answer 3

The given training results are:

$$
\text{Training accuracy} = 98\%
$$

$$
\text{Validation accuracy} = 85\%
$$

$$
\text{Training loss} = 0.02
$$

$$
\text{Validation loss} = 0.4
$$

These results indicate a clear **generalization gap**. The model performs very well on the training set, but significantly worse on the validation set.

The accuracy gap is:

$$
98\% - 85\% = 13\%
$$

The loss gap is:

$$
0.4 - 0.02 = 0.38
$$

This suggests that the model may be overfitting the training data: it has learned patterns that fit the training set very well, but those patterns do not generalize equally well to unseen validation data.

#### a)

The general parameter update rule for gradient descent is:

$$
\theta_{t+1} = \theta_t - \eta \nabla_\theta L(\theta_t)
$$

where $\theta_t$ are the model parameters at iteration $t$, $\eta$ is the learning rate, and $\nabla_\theta L(\theta_t)$ is the gradient of the loss with respect to the parameters.

For mini-batch gradient descent, the true gradient is approximated by a mini-batch gradient:

$$
g_B =
\frac{1}{B}
\sum_{i=1}^{B}
\nabla_\theta L_i(\theta)
$$

where $B$ is the batch size.

Therefore, the update becomes:

$$
\theta_{t+1} = \theta_t - \eta g_B
$$

* **Learning rate is too high**

If the learning rate $\eta$ is too high, each update step is too large:

$$
\Delta \theta = -\eta \nabla_\theta L(\theta)
$$

A large $\eta$ can cause the optimizer to overshoot the minimum instead of moving toward it smoothly.

Using a second-order Taylor approximation around $\theta_t$:

$$
L(\theta_t - \eta g)
\approx
L(\theta_t)
-
\eta \|g\|^2
+
\frac{\eta^2}{2} g^T H g
$$

where $H$ is the Hessian matrix of second derivatives.

The first-order term decreases the loss, but the second-order term grows with $\eta^2$. If $\eta$ is too large, the second-order term dominates, and the loss may increase instead of decrease.

This can cause unstable training, oscillations, or divergence. In practice, we may observe noisy or increasing training loss and poor validation performance.

Possible solutions:

$$
\eta \downarrow
$$

Reduce the learning rate, use a learning-rate scheduler, use adaptive optimizers such as Adam, or apply gradient clipping if gradients are exploding.

* **Learning rate is too small**

If the learning rate $\eta$ is too small, the update step is too small:

$$
\|\Delta \theta\|
=
\eta \|\nabla_\theta L(\theta)\|
$$

When $\eta$ is very small:

$$
\|\Delta \theta\| \approx 0
$$

so the parameters change very slowly.

This leads to slow convergence. The model may require many epochs to reach a good solution, and if training is stopped early, it may remain undertrained.

Possible solutions:

Increase the learning rate, use a learning-rate warmup, use an adaptive optimizer, or train for more epochs.

* **Batch size is too high**

For mini-batch gradient descent, the variance of the gradient estimate approximately decreases as batch size increases:

$$
\text{Var}(g_B) \approx \frac{\sigma^2}{B}
$$

where $\sigma^2$ represents the variance of individual sample gradients.

If $B$ is very large, then:

$$
\text{Var}(g_B) \downarrow
$$

so the gradient estimate becomes very stable. However, very large batch sizes can reduce the stochasticity of training.

This can make the optimizer converge to sharper minima, which may generalize worse. A sharp minimum means that a small change in parameters can cause a large increase in loss:

$$
L(\theta + \epsilon) - L(\theta)
\text{ is large}
$$

This matches the given metrics: low training loss but much higher validation loss.

Possible solutions:

Reduce the batch size, add regularization, use data augmentation, use dropout, use weight decay, or use early stopping.

* **Batch size is too small**

If the batch size is too small, the mini-batch gradient has high variance:

$$
\text{Var}(g_B) \approx \frac{\sigma^2}{B}
$$

When $B$ is small:

$$
\text{Var}(g_B) \uparrow
$$

This means the gradient direction changes significantly from one mini-batch to another.

The update rule becomes noisy:

$$
\theta_{t+1} = \theta_t - \eta g_B
$$

where $g_B$ may be a poor approximation of the true gradient.

This can make optimization unstable, cause noisy loss curves, and prevent smooth convergence. However, some noise can help generalization because it may prevent the optimizer from settling into sharp minima.

Possible solutions:

Increase the batch size moderately, reduce the learning rate, use gradient accumulation, use batch normalization, or use an adaptive optimizer such as Adam.

#### b)

To derive the relationships, define the standard confusion-matrix terms:

* $TP$: true positives
* $TN$: true negatives
* $FP$: false positives
* $FN$: false negatives

* **Precision and Recall**

Precision measures how many of the samples predicted as positive are actually positive:

$$
\text{Precision}
=
\frac{TP}{TP + FP}
$$

Recall measures how many of the actual positive samples were correctly detected:

$$
\text{Recall}
=
\frac{TP}{TP + FN}
$$

Therefore, precision is affected mainly by false positives, while recall is affected mainly by false negatives.

From the precision equation:

$$
P = \frac{TP}{TP + FP}
$$

we can solve for $FP$:

$$
P(TP + FP) = TP
$$

$$
PTP + PFP = TP
$$

$$
PFP = TP - PTP
$$

$$
FP = TP\left(\frac{1-P}{P}\right)
$$

or equivalently:

$$
FP = TP\left(\frac{1}{P} - 1\right)
$$

From the recall equation:

$$
R = \frac{TP}{TP + FN}
$$

we can solve for $FN$:

$$
R(TP + FN) = TP
$$

$$
RTP + RFN = TP
$$

$$
RFN = TP - RTP
$$

$$
FN = TP\left(\frac{1-R}{R}\right)
$$

or equivalently:

$$
FN = TP\left(\frac{1}{R} - 1\right)
$$

This shows that precision and recall are connected through $TP$, but they measure different types of errors:

$$
\text{Low precision} \Rightarrow \text{many false positives}
$$

$$
\text{Low recall} \Rightarrow \text{many false negatives}
$$

There is usually a trade-off between precision and recall. If the classification threshold is lowered, the model predicts more samples as positive. This usually increases recall but may decrease precision. If the threshold is raised, the model predicts fewer samples as positive. This usually increases precision but may decrease recall.

* **F1 score and Accuracy**

The F1 score is the harmonic mean of precision and recall:

$$
F_1
=
\frac{2PR}{P + R}
$$

Substituting:

$$
P = \frac{TP}{TP + FP}
$$

and:

$$
R = \frac{TP}{TP + FN}
$$

gives:

$$
F_1
=
\frac{
2 \cdot \frac{TP}{TP + FP} \cdot \frac{TP}{TP + FN}
}{
\frac{TP}{TP + FP} + \frac{TP}{TP + FN}
}
$$

After simplification:

$$
F_1
=
\frac{2TP}{2TP + FP + FN}
$$

Accuracy is defined as:

$$
\text{Accuracy}
=
\frac{TP + TN}{TP + TN + FP + FN}
$$

The key difference is that accuracy includes true negatives, while the F1 score does not directly include true negatives.

Therefore:

$$
F_1
=
\frac{2TP}{2TP + FP + FN}
$$

but:

$$
\text{Accuracy}
=
\frac{TP + TN}{TP + TN + FP + FN}
$$

This means there is no unique direct relationship between F1 score and accuracy unless we also know the full confusion matrix, especially the number of true negatives.

For balanced datasets, accuracy can be informative because both classes are represented similarly. However, for imbalanced datasets, accuracy may be misleading.

For example, if the negative class is much larger than the positive class, a model can achieve high accuracy by correctly predicting many true negatives, even if it performs poorly on the positive class.

In contrast, the F1 score focuses on performance on the positive class by combining precision and recall. Therefore, F1 is usually more informative than accuracy when the dataset is imbalanced or when false positives and false negatives are more important than true negatives.